# Laboratorio 6 - Task 1


## 1. Q-Learning tabular y DQN

LunarLanderContinuous-v2 usa dos acciones continuas: la fuerza de cada propulsor. Esto significa que hay infinitas acciones posibles. Q-Learning tabular no sirve porque tendría que guardar un valor $Q(s,a)$ para cada estado y cada acción posible.

Tanto Q-Learning como DQN necesitan calcular $\arg\max_a Q(s,a)$ para elegir la mejor acción. Esto es fácil cuando hay pocas acciones discretas, pero no cuando $a$ es un vector continuo. DQN representa una política indirecta: primero calcula valores Q y luego elige el mayor. Para este caso es más natural representar directamente una política continua que genere acciones.

## 2. Gradiente de la política Gaussiana

La política es una Gaussiana cuya media $\mu_\theta(s)$ sale de una red neuronal. Con $\sigma$ fija:

$$\ln\pi_\theta(a\mid s)=-\frac{d}{2}\ln(2\pi\sigma^2)-\frac{\lVert a-\mu_\theta(s)\rVert^2}{2\sigma^2}.$$

El primer término no depende de $\theta$. La parte que sí depende de $\theta$ es la media de la red. Por eso:

$$\nabla_\theta\ln\pi_\theta(a\mid s)=\frac{1}{\sigma^2}(\nabla_\theta\mu_\theta(s))^T(a-\mu_\theta(s)).$$

En simple: se compara la acción tomada con la media que propuso la red y se ajustan los pesos de la red en esa dirección.

## 3. REINFORCE con línea base vs. Actor-Critic

REINFORCE con línea base usa $\hat A_t=G_t-b(S_t)$: el retorno completo menos una estimación del valor del estado. Si la línea base solo depende del estado, no agrega sesgo. Sin embargo, el retorno completo tiene mucha varianza porque depende de todo lo que ocurre hasta terminar el episodio.

Actor-Critic suele usar $\hat A_t=R_{t+1}+\gamma V_w(S_{t+1})-V_w(S_t)$. Tiene menos varianza porque aprende con información inmediata, pero puede tener sesgo si el Critic estima mal el valor de los estados.

Esperamos que Actor-Critic converja más rápido. En LunarLander hay muchas decisiones por episodio y el Critic da una señal de aprendizaje más frecuente. REINFORCE es más simple, pero suele aprender de forma más inestable.

## 4. Suavidad de acciones en el exoesqueleto

Se puede castigar en la recompensa que la acción cambie demasiado entre un instante y otro:

$$r'_t=r_t-\lambda\lVert a_t-a_{t-1}\rVert^2.$$

También conviene limitar la fuerza máxima y la rapidez con que puede cambiar. La política puede recibir la acción anterior y generar solo un cambio pequeño: $a_t=a_{t-1}+\Delta a_t$.

La elección no cambia: Actor-Critic sigue siendo una mejor opción inicial porque normalmente necesita menos datos. Aun así, en un sistema real se deben usar límites de seguridad además de la recompensa.